# 🗺️ PERUPETRO · Mapa de lotes y variables geográficas

Descarga los polígonos de los lotes del **mapa oficial de PERUPETRO en ArcGIS Online** y calcula, para cada lote:

| Variable | Fuente |
|---|---|
| Área (km²), centroide | Polígono del lote (PERUPETRO) |
| Departamento (principal y % por departamento; los lotes en el mar se marcan ZÓCALO) | Límites departamentales (geoBoundaries, basado en INEI) |
| Cuenca sedimentaria | Capa de cuencas de PERUPETRO (o atributo del lote) |
| Distancia al Oleoducto Norperuano | Capa del mapa de PERUPETRO, servicio público elegido u OpenStreetMap |
| Distancia a Iquitos y al río grande más cercano | HydroRIVERS (caudal) + Natural Earth (nombre) |
| % del lote dentro de Áreas Naturales Protegidas | Servicio SERNANP elegido u OpenStreetMap |
| Comunidades nativas dentro y a menos de 5 km | Servicio público elegido (BDPI – Ministerio de Cultura u otro) |

**Uso:** ejecuta los bloques en orden. El **Bloque 2** lista las capas encontradas y propone una selección; revísala antes del Bloque 3.

**Resultados en Drive:**
- `DATA/PP_LOTES_GEO.csv`: una fila por lote con todas las variables (la usa el notebook de producción);
- `DATA/MAPA/`: GeoJSON de cada capa y `PP_mapa_lotes.html` (mapa interactivo);
- `CONTROL/CONTROL PP mapa capas.csv`: qué se descargó, de dónde y cuántas entidades.

Visor original: https://perupetro.maps.arcgis.com/apps/webappviewer/index.html?id=cc883917c55a4264a01debfa02585e2e

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN, DRIVE Y FUNCIONES (ejecutar una vez por sesión) ═══
!pip install geopandas pyogrio folium requests ipywidgets google-api-python-client --quiet
import os, re, io, json, time, logging, unicodedata
from datetime import date
import requests
import pandas as pd
import geopandas as gpd
import ipywidgets as w
from IPython.display import display
from shapely.geometry import shape, Point, LineString, MultiLineString, Polygon, MultiPolygon

DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
ruta = "/content/PP_Mapa"; os.makedirs(ruta, exist_ok=True)

# ── Google Drive ─────────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def carpeta(nombre, padre, existentes):
    if nombre not in existentes:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [padre],
                                                        "mimeType": "application/vnd.google-apps.folder"},
                                                  fields="id", supportsAllDrives=True).execute()["id"]
    return existentes[nombre]

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
MAPA_FOLDER_ID = carpeta("MAPA", DRIVE_FOLDER_ID, datos_drive)
mapa_drive = archivos_en(MAPA_FOLDER_ID)
print(f"☁️  Drive conectado · archivos en DATA/MAPA: {len(mapa_drive)}")

# ── Funciones: ArcGIS REST, fuentes externas e indicadores ──────────────────

# ── Conexión ─────────────────────────────────────────────────────────────────
SES = requests.Session()
SES.headers["User-Agent"] = "Mozilla/5.0 (proyecto academico; consulta de datos publicos)"

def get_json(url, params=None, post=False, intentos=4):
    """GET/POST a un servicio REST de ArcGIS; reintenta y avisa los errores que ArcGIS manda con código 200."""
    params = dict(params or {})
    params.setdefault("f", "json")
    for i in range(1, intentos + 1):
        try:
            r = (SES.post(url, data=params, timeout=120) if post else SES.get(url, params=params, timeout=120))
            r.raise_for_status()
            js = r.json()
            if isinstance(js, dict) and "error" in js:
                cod = js["error"].get("code")
                if cod in (498, 499, 403):
                    raise PermissionError(f"requiere inicio de sesión ({cod})")
                raise RuntimeError(js["error"].get("message", js["error"]))
            return js
        except PermissionError:
            raise
        except Exception as e:
            if i == intentos:
                raise
            time.sleep(2 ** i)

def norm_txt(t):
    t = unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().upper()

def slug(t):
    return re.sub(r"[^A-Z0-9]+", "_", norm_txt(t)).strip("_")[:60] or "CAPA"

# ── 1. Descubrimiento: aplicación → mapa web → capas ────────────────────────
def _item(portal, item_id):
    return get_json(f"{portal}/sharing/rest/content/items/{item_id}")

def _item_data(portal, item_id):
    return get_json(f"{portal}/sharing/rest/content/items/{item_id}/data")

def mapa_web_de_app(portal, app_id):
    """Devuelve (id del mapa web, título) a partir del id de la aplicación (Web AppBuilder, Instant Apps, etc.)."""
    info = _item(portal, app_id)
    if info.get("type") == "Web Map":
        return app_id, info.get("title", "")
    data = _item_data(portal, app_id)
    candidatos = []
    def buscar(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if k in ("itemId", "webmap", "webMap", "id") and isinstance(v, str) and re.fullmatch(r"[0-9a-f]{32}", v):
                    candidatos.append(v)
                buscar(v)
        elif isinstance(o, list):
            for v in o:
                buscar(v)
    if isinstance(data.get("map"), dict) and data["map"].get("itemId"):
        candidatos.insert(0, data["map"]["itemId"])
    buscar(data)
    for c in dict.fromkeys(candidatos):
        if c == app_id:
            continue
        try:
            inf = _item(portal, c)
        except Exception:
            continue
        if inf.get("type") == "Web Map":
            return c, inf.get("title", "")
    raise RuntimeError("No encontré el mapa web dentro de la aplicación.")

def capas_de_servicio(url, ruta_titulo=""):
    """Expande un servicio MapServer/FeatureServer o una capa suelta en una lista de capas consultables."""
    url = url.rstrip("/")
    out = []
    if re.search(r"/(MapServer|FeatureServer)$", url, re.I):
        js = get_json(url)
        for l in js.get("layers", []) + js.get("tables", []):
            if l.get("subLayerIds"):
                continue                                  # grupo: sus hijas vienen en la misma lista
            out += capas_de_servicio(f"{url}/{l['id']}", ruta_titulo)
        return out
    js = get_json(url)
    if js.get("type") in ("Group Layer",) or js.get("subLayers") and not js.get("geometryType"):
        for s in js.get("subLayers", []):
            out += capas_de_servicio(f"{url.rsplit('/', 1)[0]}/{s['id']}", ruta_titulo)
        return out
    out.append({"CAPA": js.get("name", ""), "GRUPO": ruta_titulo, "URL": url, "TIPO": js.get("type", ""),
                "GEOMETRIA": js.get("geometryType", ""), "CAMPOS": ", ".join(f["name"] for f in js.get("fields") or []),
                "MAX_REG": js.get("maxRecordCount") or 1000,
                "PAGINA": bool((js.get("advancedQueryCapabilities") or {}).get("supportsPagination"))})
    return out

def capas_del_mapa(portal, webmap_id):
    data = _item_data(portal, webmap_id)
    out = []
    def recorrer(capas, ruta=""):
        for c in capas:
            titulo = c.get("title") or c.get("id") or ""
            r = f"{ruta} / {titulo}" if ruta else titulo
            if c.get("layers") and c.get("layerType") == "GroupLayer":
                recorrer(c["layers"], r); continue
            if c.get("featureCollection"):
                for k, l in enumerate(c["featureCollection"].get("layers", [])):
                    ld = l.get("layerDefinition", {})
                    out.append({"CAPA": ld.get("name") or titulo, "GRUPO": r, "URL": f"featureCollection:{c.get('id')}:{k}",
                                "TIPO": "Colección incrustada", "GEOMETRIA": ld.get("geometryType", ""),
                                "CAMPOS": ", ".join(f["name"] for f in ld.get("fields") or []),
                                "MAX_REG": None, "PAGINA": False, "_FC": l})
                continue
            url = c.get("url")
            if not url and c.get("itemId"):
                try:
                    url = _item(portal, c["itemId"]).get("url")
                except Exception:
                    url = None
            if not url:
                out.append({"CAPA": titulo, "GRUPO": r, "URL": "", "TIPO": c.get("layerType", ""), "GEOMETRIA": "",
                            "CAMPOS": "", "MAX_REG": None, "PAGINA": False, "ERROR": "sin URL"}); continue
            try:
                hijas = capas_de_servicio(url, r)
                if c.get("layers") and re.search(r"/MapServer$", url.rstrip("/"), re.I):
                    ids = {str(l.get("id")) for l in c["layers"]}
                    hijas = [h for h in hijas if h["URL"].rsplit("/", 1)[1] in ids] or hijas
                out.extend(hijas)
            except Exception as e:
                out.append({"CAPA": titulo, "GRUPO": r, "URL": url, "TIPO": c.get("layerType", ""), "GEOMETRIA": "",
                            "CAMPOS": "", "MAX_REG": None, "PAGINA": False, "ERROR": str(e)[:200]})
    recorrer(data.get("operationalLayers", []))
    return out

def buscar_arcgis(consulta, n=15, portal="https://www.arcgis.com"):
    """Busca servicios públicos en ArcGIS Online (para ANP, comunidades nativas, oleoducto…)."""
    q = f'({consulta}) AND (type:"Feature Service" OR type:"Map Service")'
    js = get_json(f"{portal}/sharing/rest/search", {"q": q, "num": n, "sortField": "numviews", "sortOrder": "desc"})
    return [{"TITULO": r.get("title", ""), "DUEÑO": r.get("owner", ""), "TIPO": r.get("type", ""),
             "URL": r.get("url") or "", "ID": r.get("id"), "VISTAS": r.get("numViews", 0)}
            for r in js.get("results", []) if r.get("url")]

# ── 2. Descarga de una capa ──────────────────────────────────────────────────
def _anillos_a_poligono(rings):
    """Geometría Esri (anillos) → Polygon/MultiPolygon: horario = exterior, antihorario = hueco."""
    ext, huecos = [], []
    for r in rings:
        if len(r) < 4:
            continue
        p = Polygon(r)
        (huecos if p.exterior.is_ccw else ext).append(p)
    if not ext:                                  # anillos mal orientados: todos como exteriores
        ext, huecos = huecos, []
    polys = []
    for e in ext:
        hs = [h.exterior.coords for h in huecos if e.contains(h.representative_point())]
        polys.append(Polygon(e.exterior.coords, hs))
    return polys[0] if len(polys) == 1 else MultiPolygon(polys)

def esri_a_shapely(g):
    if not g:
        return None
    if "x" in g:
        return None if g["x"] is None else Point(g["x"], g["y"])
    if "points" in g:
        from shapely.geometry import MultiPoint
        return MultiPoint(g["points"])
    if "paths" in g:
        return LineString(g["paths"][0]) if len(g["paths"]) == 1 else MultiLineString(g["paths"])
    if "rings" in g:
        return _anillos_a_poligono(g["rings"])
    return None

def gdf_desde_esri(features, wkid):
    filas = [dict(f.get("attributes") or {}) for f in features]
    geoms = [esri_a_shapely(f.get("geometry")) for f in features]
    crs = "EPSG:3857" if wkid in (102100, 102113, 3857, 900913) else f"EPSG:{wkid or 4326}"
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=crs).to_crs(4326)

def descargar_capa(capa, aviso=print):
    """Descarga todas las entidades de una capa en EPSG:4326 (GeoJSON si el servidor lo da; si no, JSON de Esri)."""
    if capa.get("_FC"):
        fs = capa["_FC"].get("featureSet", {})
        wkid = ((capa["_FC"].get("layerDefinition", {}).get("extent") or {}).get("spatialReference") or {}).get("wkid") \
               or (fs.get("spatialReference") or {}).get("wkid") or 102100
        return gdf_desde_esri(fs.get("features", []), wkid)
    url = capa["URL"].rstrip("/") + "/query"
    base = {"where": "1=1", "outFields": "*", "outSR": 4326, "returnGeometry": "true"}
    total = get_json(url, {**base, "returnCountOnly": "true"}, post=True).get("count")
    paso = int(capa.get("MAX_REG") or 1000)
    wk = lambda js: (js.get("spatialReference") or {}).get("wkid", 4326)
    partes = []
    def por_ids():
        ids = sorted(get_json(url, {"where": "1=1", "returnIdsOnly": "true"}, post=True).get("objectIds") or [])
        for i in range(0, len(ids), paso):
            js = get_json(url, {**base, "where": "", "objectIds": ",".join(map(str, ids[i:i + paso]))}, post=True)
            partes.append(gdf_desde_esri(js.get("features", []), wk(js)))
    if capa.get("PAGINA"):
        off = 0
        while True:
            js = get_json(url, {**base, "resultOffset": off, "resultRecordCount": paso}, post=True)
            feats = js.get("features", [])
            if feats:
                partes.append(gdf_desde_esri(feats, wk(js)))
            off += len(feats)
            if not feats or (total is not None and off >= total) or (total is None and not js.get("exceededTransferLimit")):
                break
    elif total and total > paso:
        por_ids()
    else:
        js = get_json(url, base, post=True)
        if js.get("exceededTransferLimit"):
            por_ids()
        else:
            partes.append(gdf_desde_esri(js.get("features", []), wk(js)))
    gdf = pd.concat(partes, ignore_index=True) if partes else gpd.GeoDataFrame(geometry=[], crs=4326)
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=4326)
    if total is not None and len(gdf) != total:
        aviso(f"   ⚠️ {capa['CAPA']}: el servidor declara {total} entidades y llegaron {len(gdf)}")
    return gdf

# ── 3. Fuentes externas ──────────────────────────────────────────────────────
GEOBOUNDARIES_API = "https://www.geoboundaries.org/api/current/gbOpen/PER/ADM1/"
NE_RIOS = "https://naciscdn.org/naturalearth/10m/physical/ne_10m_rivers_lake_centerlines.zip"
HYDRORIVERS_SA = "https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_sa_shp.zip"
OVERPASS = "https://overpass-api.de/api/interpreter"
BBOX_PERU = (-81.5, -18.5, -68.5, 0.1)                     # lon_min, lat_min, lon_max, lat_max

def bajar(url, destino, aviso=print):
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return destino
    with SES.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(destino + ".part", "wb") as fh:
            for b in r.iter_content(1 << 20):
                fh.write(b)
    os.replace(destino + ".part", destino)
    return destino

def departamentos(carpeta):
    js = SES.get(GEOBOUNDARIES_API, timeout=60).json()
    ruta = bajar(js["gjDownloadURL"], os.path.join(carpeta, "geoBoundaries-PER-ADM1.geojson"))
    g = gpd.read_file(ruta).to_crs(4326)
    g["DEPARTAMENTO"] = g["shapeName"].map(norm_txt).str.replace(r"^(REGION|DEPARTAMENTO) (DE )?", "", regex=True)
    g["DEPARTAMENTO"] = g["DEPARTAMENTO"].replace({"LIMA PROVINCE": "LIMA", "PROVINCIA DE LIMA": "LIMA",
                                                   "LIMA METROPOLITANA": "LIMA", "CALLAO PROVINCE": "CALLAO",
                                                   "PROVINCIA CONSTITUCIONAL DEL CALLAO": "CALLAO"})
    return g.dissolve("DEPARTAMENTO", as_index=False)[["DEPARTAMENTO", "geometry"]]

def rios_hydrorivers(carpeta, caudal_min):
    ruta = bajar(HYDRORIVERS_SA, os.path.join(carpeta, "HydroRIVERS_v10_sa_shp.zip"))
    g = gpd.read_file(f"zip://{ruta}", bbox=BBOX_PERU)
    return g[g["DIS_AV_CMS"] >= caudal_min][["HYRIV_ID", "DIS_AV_CMS", "ORD_FLOW", "geometry"]].to_crs(4326)

def rios_natural_earth(carpeta):
    ruta = bajar(NE_RIOS, os.path.join(carpeta, "ne_10m_rivers_lake_centerlines.zip"))
    g = gpd.read_file(f"zip://{ruta}", bbox=BBOX_PERU).to_crs(4326)
    g["RIO"] = g["name"].fillna(g.get("name_es")).fillna("")
    return g[["RIO", "geometry"]]

def overpass(consulta):
    r = SES.post(OVERPASS, data={"data": consulta}, timeout=600)
    r.raise_for_status()
    return r.json()

def _osm_a_gdf(js, tipo):
    filas, geoms = [], []
    for e in js.get("elements", []):
        t = e.get("tags", {})
        if e["type"] == "way" and e.get("geometry"):
            coords = [(p["lon"], p["lat"]) for p in e["geometry"]]
            if tipo == "linea":
                g = LineString(coords)
            elif len(coords) >= 4 and coords[0] == coords[-1]:
                g = Polygon(coords)
            else:
                continue
        elif e["type"] == "relation" and e.get("members") and tipo == "poligono":
            ext = [[(p["lon"], p["lat"]) for p in m.get("geometry", [])] for m in e["members"]
                   if m.get("role") in ("outer", "") and m.get("geometry")]
            from shapely.ops import linemerge, polygonize, unary_union
            polys = list(polygonize(unary_union([LineString(c) for c in ext if len(c) >= 2])))
            if not polys:
                continue
            g = unary_union(polys)
        else:
            continue
        filas.append({"NOMBRE": t.get("name", ""), "OSM_ID": f"{e['type']}/{e['id']}", **{k: t.get(k, "") for k in
                     ("protect_class", "protection_title", "operator", "substance")}})
        geoms.append(g)
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=4326)

def anp_osm():
    q = """[out:json][timeout:500];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    (way["boundary"="protected_area"](area.pe);relation["boundary"="protected_area"](area.pe);
     way["leisure"="nature_reserve"](area.pe);relation["leisure"="nature_reserve"](area.pe););out geom;"""
    return _osm_a_gdf(overpass(q), "poligono")

def oleoducto_osm():
    q = """[out:json][timeout:300];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    (way["man_made"="pipeline"]["substance"~"oil|crude",i](area.pe););out geom;"""
    return _osm_a_gdf(overpass(q), "linea")

# ── 4. Indicadores por lote ──────────────────────────────────────────────────
CRS_AREA = "ESRI:102033"        # Albers equivalente de Sudamérica (áreas en km²)
CRS_DIST = "EPSG:32718"         # UTM 18S (distancias en km; error < 1 % en la Amazonía peruana)
IQUITOS = Point(-73.2538, -3.7491)

def lote_clave(t):
    """'LOTE 192' → '192'; 'Lote Z - 2B' → 'Z-2B'; '1 AB' → '1-AB' (misma clave que el notebook de producción)."""
    s = norm_txt(t)
    s = re.sub(r"^(LOTE|BLOQUE|AREA|CONTRATO)\s+", "", s)
    s = re.sub(r"\s*-\s*", "-", s).replace(" ", "")
    s = re.sub(r"^([A-Z]{1,2})(\d)", r"\1-\2", s) if re.match(r"^(Z|XX?|CO)\d", s) else s
    s = re.sub(r"^(\d+)([A-Z]+)$", r"\1-\2", s) if re.match(r"^\d+[A-Z]{2}$", s) else s
    return s

def campo_lote(gdf):
    cands = [c for c in gdf.columns if c != "geometry"]
    for pat in (r"^LOTE$", r"LOTE", r"^(NOMBRE|NAME|NOM)$", r"NOMBRE|NAME|CONTRATO|BLOQUE"):
        for c in cands:
            if re.search(pat, norm_txt(c)) and gdf[c].astype(str).str.len().gt(0).mean() > 0.5:
                return c
    return None

def campo(gdf, *patrones):
    for p in patrones:
        for c in gdf.columns:
            if c != "geometry" and re.search(p, norm_txt(c)):
                return c
    return None

def reparto_por_poligonos(lotes, capa, nombre_col, prefijo):
    """% del área de cada lote dentro de cada polígono de `capa` → principal, detalle y suma."""
    if capa is None or capa.empty:
        return pd.DataFrame(index=lotes.index)
    a = lotes[["geometry"]].to_crs(CRS_AREA)
    b = capa[[nombre_col, "geometry"]].to_crs(CRS_AREA)
    b = b[b.geometry.notna() & ~b.geometry.is_empty].copy()
    b["geometry"] = b.geometry.buffer(0)
    a = a.copy(); a["geometry"] = a.geometry.buffer(0); a["_ID"] = a.index
    inter = gpd.overlay(a, b, how="intersection", keep_geom_type=True)
    inter["_KM2"] = inter.geometry.area / 1e6
    area = a.geometry.area / 1e6
    res = {}
    for i, g in inter.groupby("_ID"):
        s = g.groupby(nombre_col)["_KM2"].sum().sort_values(ascending=False)
        pct = (s / area.loc[i] * 100).round(1)
        res[i] = {f"{prefijo}": s.index[0], f"{prefijo}_PCT": min(pct.iloc[0], 100.0),
                  f"{prefijo}_DETALLE": "; ".join(f"{k} {v:.1f}%" for k, v in pct.items() if v >= 0.1),
                  f"{prefijo}_N": int((pct >= 0.1).sum()), f"{prefijo}_PCT_TOTAL": min(round(pct.sum(), 1), 100.0)}
    return pd.DataFrame.from_dict(res, orient="index").reindex(lotes.index)

def distancia_km(lotes, capa, nombre_col=None):
    """Distancia mínima (km) de cada lote a la capa (0 si la toca) y, si hay nombre, el elemento más cercano."""
    if capa is None or capa.empty:
        return pd.Series(index=lotes.index, dtype=float), pd.Series(index=lotes.index, dtype=object)
    a = lotes.to_crs(CRS_DIST)
    b = capa[capa.geometry.notna()].to_crs(CRS_DIST)
    j = gpd.sjoin_nearest(a[["geometry"]], b[[c for c in [nombre_col] if c] + ["geometry"]], how="left",
                          distance_col="_D")
    j = j[~j.index.duplicated()]
    return (j["_D"] / 1000).round(1), (j[nombre_col] if nombre_col else pd.Series(index=lotes.index, dtype=object))

def comunidades(lotes, ccnn, nombre_col, radio_km=5):
    """Comunidades nativas dentro del lote y a menos de `radio_km` km (cuenta y nombres)."""
    if ccnn is None or ccnn.empty:
        return pd.DataFrame(index=lotes.index)
    a = lotes[["geometry"]].to_crs(CRS_DIST)
    b = ccnn[[nombre_col, "geometry"]].to_crs(CRS_DIST)
    b = b[b.geometry.notna()].copy()
    pol = b.geom_type.isin(["Polygon", "MultiPolygon"])
    b.loc[pol, "geometry"] = b.loc[pol, "geometry"].buffer(0)
    out = {}
    dentro = gpd.sjoin(a, b, predicate="intersects")
    cerca = gpd.sjoin(gpd.GeoDataFrame(geometry=a.geometry.buffer(radio_km * 1000), crs=a.crs), b, predicate="intersects")
    for i in lotes.index:
        d = dentro[dentro.index == i][nombre_col].dropna().astype(str).unique()
        c = cerca[cerca.index == i][nombre_col].dropna().astype(str).unique()
        out[i] = {"CCNN_DENTRO_N": len(d), f"CCNN_{radio_km}KM_N": len(c),
                  "CCNN_DENTRO": "; ".join(sorted(d)[:40]) + (" …" if len(d) > 40 else "")}
    return pd.DataFrame.from_dict(out, orient="index")

print("✅ Funciones listas")

In [ ]:
# ═══ BLOQUE 2 · DESCUBRIR CAPAS Y ELEGIR FUENTES ═══
PORTAL = "https://perupetro.maps.arcgis.com"
APP_ID = "cc883917c55a4264a01debfa02585e2e"          # visor "mapa de lotes" de PERUPETRO
URLS_EXTRA = []        # opcional: URLs de servicios MapServer/FeatureServer adicionales (una por elemento)
BUSQUEDAS = {          # búsquedas en ArcGIS Online (respaldo y capas que no son de PERUPETRO)
    "LOTES": "lotes hidrocarburos Peru",
    "LOTES2": "lotes de contrato PERUPETRO",
    "ANP": "areas naturales protegidas SERNANP",
    "CCNN": "comunidades nativas Peru",
    "OLEODUCTO": "oleoducto norperuano",
}
GEOM_TEMA = {"LOTES": ("Polygon",), "LOTES2": ("Polygon",), "ANP": ("Polygon",), "CCNN": ("Polygon", "Point"), "OLEODUCTO": ("Polyline",)}

print("🔎 Leyendo el visor de PERUPETRO…")
WEBMAP_ID, WEBMAP_TIT = mapa_web_de_app(PORTAL, APP_ID)
print(f"   Mapa web: «{WEBMAP_TIT}» ({WEBMAP_ID})")
capas_pp = capas_del_mapa(PORTAL, WEBMAP_ID)
for u in URLS_EXTRA:
    try:
        capas_pp += capas_de_servicio(u, "EXTRA")
    except Exception as e:
        print(f"   ⚠️ {u}: {e}")
for c in capas_pp:
    c["FUENTE"], c["TEMA"] = "PERUPETRO", ""

capas_ext = []
for tema, q in BUSQUEDAS.items():
    try:
        res = buscar_arcgis(q)
    except Exception as e:
        print(f"   ⚠️ búsqueda {tema}: {e}"); res = []
    for r in res[:8]:
        try:
            for c in capas_de_servicio(r["URL"], r["TITULO"]):
                if any(g in c["GEOMETRIA"] for g in GEOM_TEMA[tema]) and c["URL"] not in {x["URL"] for x in capas_ext}:
                    c.update({"FUENTE": f"ArcGIS Online · {r['DUEÑO']}", "TEMA": tema.rstrip("2")})
                    capas_ext.append(c)
        except Exception as e:
            pass

def texto(c):
    return norm_txt(f"{c['CAPA']} {c['GRUPO']} {c['URL']} {c['FUENTE']}")

def etiqueta(c):
    g = c["GEOMETRIA"].replace("esriGeometry", "")
    return f"{c['CAPA']}  [{g}]  ·  {c['GRUPO'][:50]}  ·  {c['FUENTE'][:40]}"

capas = capas_pp + capas_ext
for i, c in enumerate(capas):
    c["N"] = i
POR_N_ = {c["N"]: c for c in capas}
cat = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in capas])
pd.set_option("display.max_colwidth", 70)
print(f"\n📋 Capas de PERUPETRO: {len(capas_pp)}")
display(cat[cat["FUENTE"] == "PERUPETRO"][["N", "CAPA", "GRUPO", "GEOMETRIA", "TIPO"] + (["ERROR"] if "ERROR" in cat else [])])
print(f"\n📋 Candidatas externas: {len(capas_ext)}")
if capas_ext:
    display(cat[cat["FUENTE"] != "PERUPETRO"][["N", "TEMA", "CAPA", "GRUPO", "GEOMETRIA", "FUENTE"]])

# ── Propuesta automática (revísala) ─────────────────────────────────────────
fallidas = [c for c in capas_pp if c.get("ERROR")]
if fallidas:
    print("\n⚠️ Capas del visor de PERUPETRO que no responden (también fallan en el navegador, p. ej. MapaBase_Set2019):")
    for c in fallidas:
        print(f"   · {c['CAPA']}: {c['ERROR']}  {c['URL']}")
con_geo = [c for c in capas_pp if c["GEOMETRIA"] and not c.get("ERROR")]
poligonos = [c for c in con_geo if "Polygon" in c["GEOMETRIA"]]
es_lote = lambda c: re.search(r"LOTE|CONTRAT|BLOQUE|ASIGNAC|EXPLOTAC|EXPLORAC", texto(c)) and not re.search(r"CUENCA|SISMIC", texto(c))
def mejor(cands, patron, extra=r"SERNANP|CULTURA|MINCU|GOB.PE|MINEM|OSINERG"):
    puntaje = lambda c: (bool(re.search(patron, texto(c))) * 2 + bool(re.search(extra, texto(c))))
    cands = sorted(cands, key=puntaje, reverse=True)
    return cands[0]["N"] if cands and puntaje(cands[0]) >= 2 else None

OSM = "OpenStreetMap (respaldo)"; NINGUNA = "(ninguna)"
_l = {"layout": w.Layout(width="95%")}
sel_capas = w.SelectMultiple(options=[(etiqueta(c), c["N"]) for c in con_geo], value=[c["N"] for c in con_geo],
                             rows=min(12, max(3, len(con_geo))), description="Descargar", **_l)
lotes_ext = [c for c in capas_ext if c["TEMA"] == "LOTES"]
_lv = [c["N"] for c in poligonos if es_lote(c)]
if not _lv:                                   # el visor no da lotes: mejor candidata externa
    _m = mejor(lotes_ext, r"LOTE", r"PERUPETRO|MINEM|OSINERG|GOB.PE|INGEMMET|GEOCATMIN")
    _lv = [_m] if _m is not None else []
    print("\n⚠️ El visor de PERUPETRO no entregó polígonos de lotes; propongo:",
          POR_N_[_lv[0]]["CAPA"] + " · " + POR_N_[_lv[0]]["FUENTE"] if _lv else "ninguna (agrega una URL en URLS_EXTRA)")
sel_lotes = w.SelectMultiple(options=[(etiqueta(c), c["N"]) for c in poligonos + lotes_ext], value=_lv,
                             rows=min(10, max(3, len(poligonos) + len(lotes_ext))), description="Son lotes", **_l)
_cu = mejor(poligonos, r"CUENCA")
dd_cuenca = w.Dropdown(options=[(NINGUNA, None)] + [(etiqueta(c), c["N"]) for c in poligonos], value=_cu,
                       description="Cuencas", **_l)
def dd_tema(tema, patron, respaldo):
    pp = [c for c in con_geo if any(g in c["GEOMETRIA"] for g in GEOM_TEMA[tema])]
    ex = [c for c in capas_ext if c["TEMA"] == tema]
    ops = [(etiqueta(c), c["N"]) for c in pp + ex] + [(respaldo, respaldo)]
    val = mejor(pp, patron) if mejor(pp, patron) is not None else mejor(ex, patron)
    return w.Dropdown(options=ops, value=val if val is not None else respaldo, description=tema, **_l)
dd_anp = dd_tema("ANP", r"AREA(S)? NATURAL|PROTEGID|\bANP\b", OSM)
dd_ccnn = dd_tema("CCNN", r"COMUNIDAD(ES)? NATIVA|CC\.?NN|PUEBLOS? INDIGENA", NINGUNA)
dd_onp = dd_tema("OLEODUCTO", r"OLEODUCTO|ONP|NORPERUANO|DUCTO", OSM)
txt_caudal = w.IntText(value=300, description="Río grande: caudal medio ≥ (m³/s)", style={"description_width": "260px"})
display(w.VBox([w.HTML("<b>Capas de PERUPETRO a descargar</b> (Ctrl/Cmd + clic para cambiar la selección)"), sel_capas,
                w.HTML("<b>¿Cuáles son polígonos de lotes?</b>"), sel_lotes,
                w.HTML("<b>Fuentes para los indicadores</b>"), dd_cuenca, dd_onp, dd_anp, dd_ccnn, txt_caudal]))

cat_ruta = os.path.join(ruta, "CONTROL PP mapa capas.csv")
cat.assign(FECHA_CONSULTA=date.today().strftime("%d/%m/%Y")).to_csv(cat_ruta, index=False, encoding="utf-8-sig")
subir(cat_ruta, DRIVE_FOLDER_CONTROL_ID, control_drive)
print("☁️  Catálogo de capas guardado en CONTROL. Revisa la selección y ejecuta el Bloque 3.")

In [ ]:
# ═══ BLOQUE 3 · DESCARGA Y VARIABLES POR LOTE ═══
LOTES_ALIAS_GEOMETRIA = {"1-AB": "192"}   # lote histórico → polígono actual del mismo campo
POR_N = {c["N"]: c for c in capas}
log = []

def guardar_geojson(gdf, nombre):
    p = os.path.join(ruta, nombre)
    gdf.to_file(p, driver="GeoJSON")
    subir(p, MAPA_FOLDER_ID, mapa_drive, mime="application/geo+json")

def capa_gdf(n, cache={}):
    if n in cache:
        return cache[n]
    c = POR_N[n]
    print(f"⬇️  {c['CAPA']} …", end=" ")
    try:
        g = descargar_capa(c)
        print(f"{len(g)} entidades")
        guardar_geojson(g, f"{'PP' if c['FUENTE'] == 'PERUPETRO' else 'EXT'}_{slug(c['CAPA'])}.geojson")
        log.append({"CAPA": c["CAPA"], "URL": c["URL"], "FUENTE": c["FUENTE"], "ENTIDADES": len(g), "ESTADO": "OK"})
    except Exception as e:
        print(f"❌ {e}"); g = None
        log.append({"CAPA": c["CAPA"], "URL": c["URL"], "FUENTE": c["FUENTE"], "ENTIDADES": 0, "ESTADO": f"ERROR: {e}"[:200]})
    cache[n] = g
    return g

for n in sel_capas.value:
    capa_gdf(n)

# ── 1. Lotes ─────────────────────────────────────────────────────────────────
partes = []
for n in sel_lotes.value:
    g = capa_gdf(n)
    if g is None or g.empty:
        continue
    f_lote = campo_lote(g)
    if not f_lote:
        print(f"⚠️ {POR_N[n]['CAPA']}: no encontré el campo con el nombre del lote. Campos: {list(g.columns)}"); continue
    f_op = campo(g, r"OPERADOR|CONTRATISTA|EMPRESA|COMPANIA|TITULAR")
    f_est = campo(g, r"ESTADO|SITUACION|FASE|ETAPA")
    f_cu = campo(g, r"CUENCA")
    partes.append(gpd.GeoDataFrame({
        "LOTE": g[f_lote].map(lote_clave), "LOTE_MAPA": g[f_lote].astype(str),
        "OPERADOR_MAPA": g[f_op].astype(str) if f_op else "", "ESTADO_MAPA": g[f_est].astype(str) if f_est else "",
        "CUENCA_ATRIBUTO": g[f_cu].astype(str) if f_cu else "", "CAPA": POR_N[n]["CAPA"]}, geometry=g.geometry, crs=4326))
if not partes:
    raise RuntimeError("No hay capas de lotes seleccionadas o no se pudo leer ninguna.")
lotes = pd.concat(partes, ignore_index=True)
lotes = lotes[lotes.geometry.notna() & ~lotes.geometry.is_empty & lotes["LOTE"].ne("")]
lotes["geometry"] = lotes.geometry.buffer(0)
lotes = lotes.dissolve("LOTE", aggfunc={"LOTE_MAPA": "first", "OPERADOR_MAPA": "first", "ESTADO_MAPA": "first",
                                       "CUENCA_ATRIBUTO": "first", "CAPA": lambda s: " | ".join(sorted(set(s)))}).reset_index()
lotes["GEOMETRIA_DE"] = lotes["LOTE"]
for hist, actual in LOTES_ALIAS_GEOMETRIA.items():
    if hist not in set(lotes["LOTE"]) and actual in set(lotes["LOTE"]):
        f = lotes[lotes["LOTE"] == actual].copy(); f["LOTE"] = hist; f["GEOMETRIA_DE"] = actual
        lotes = pd.concat([lotes, f], ignore_index=True)
lotes = gpd.GeoDataFrame(lotes, geometry="geometry", crs=4326)
print(f"\n🧩 Lotes con polígono: {lotes['LOTE'].nunique()}")

# ── 2. Capas externas ────────────────────────────────────────────────────────
def externa(nombre, fn):
    print(f"⬇️  {nombre} …", end=" ")
    try:
        g = fn(); print(f"{len(g)} entidades")
        guardar_geojson(g, f"EXT_{slug(nombre)}.geojson")
        log.append({"CAPA": nombre, "URL": "", "FUENTE": "externa", "ENTIDADES": len(g), "ESTADO": "OK"})
        return g
    except Exception as e:
        print(f"❌ {e}")
        log.append({"CAPA": nombre, "URL": "", "FUENTE": "externa", "ENTIDADES": 0, "ESTADO": f"ERROR: {e}"[:200]})
        return None

def elegida(dd, respaldo_fn, nombre):
    v = dd.value
    if v in (None, NINGUNA):
        return None, None
    if v == OSM:
        return externa(f"{nombre} OSM", respaldo_fn), "NOMBRE"
    g = capa_gdf(v)
    return g, (campo(g, r"^(NOMBRE|NOM_ANP|ANP_NOMB|NOMB_CCNN|NOM_COMUNI|NAME)$", r"NOMB|NOMBRE|NAME|DENOMIN") if g is not None else None)

deps = externa("Departamentos geoBoundaries", lambda: departamentos(ruta))
rios_q = externa("Rios HydroRIVERS", lambda: rios_hydrorivers(ruta, txt_caudal.value))
rios_n = externa("Rios Natural Earth", lambda: rios_natural_earth(ruta))
onp, onp_n = elegida(dd_onp, oleoducto_osm, "Oleoducto")
anp, anp_n = elegida(dd_anp, anp_osm, "ANP")
ccnn, ccnn_n = elegida(dd_ccnn, None, "Comunidades nativas")
cuencas = capa_gdf(dd_cuenca.value) if dd_cuenca.value is not None else None
cuenca_n = campo(cuencas, r"CUENCA", r"NOMB|NOMBRE|NAME") if cuencas is not None else None

# ── 3. Indicadores ───────────────────────────────────────────────────────────
geo = lotes.copy()
geo["AREA_KM2"] = (geo.to_crs(CRS_AREA).geometry.area / 1e6).round(1)
rp = geo.geometry.representative_point()
geo["LAT"], geo["LON"] = rp.y.round(5), rp.x.round(5)
if deps is not None:
    geo = geo.join(reparto_por_poligonos(geo, deps, "DEPARTAMENTO", "DEPARTAMENTO"))
    sin = geo["DEPARTAMENTO"].isna()
    geo["ZOCALO"] = sin.map({True: "SI", False: "NO"})
    if sin.any():                                    # lotes en el mar: región costera más cercana
        d, n = distancia_km(geo[sin], deps, "DEPARTAMENTO")
        geo.loc[sin, "DEPARTAMENTO"] = n.values
        geo.loc[sin, "DEPARTAMENTO_DETALLE"] = "ZÓCALO frente a " + n.astype(str).values
if cuencas is not None and cuenca_n:
    geo = geo.join(reparto_por_poligonos(geo, cuencas, cuenca_n, "CUENCA")[["CUENCA", "CUENCA_PCT"]])
else:
    geo["CUENCA"] = geo["CUENCA_ATRIBUTO"]
if onp is not None:
    geo["DIST_OLEODUCTO_KM"], _ = distancia_km(geo, onp)
geo["DIST_IQUITOS_KM"] = (geo.to_crs(CRS_DIST).distance(gpd.GeoSeries([IQUITOS], crs=4326).to_crs(CRS_DIST).iloc[0]) / 1000).round(1)
if rios_q is not None:
    geo["DIST_RIO_GRANDE_KM"], _ = distancia_km(geo, rios_q)
if rios_n is not None:
    _, geo["RIO_CERCANO"] = distancia_km(geo, rios_n, "RIO")
if anp is not None and anp_n:
    a = reparto_por_poligonos(geo, anp, anp_n, "ANP")
    geo["ANP_PCT"] = a.get("ANP_PCT_TOTAL"); geo["ANP_DETALLE"] = a.get("ANP_DETALLE")
    geo["ANP_PCT"] = geo["ANP_PCT"].fillna(0)
if ccnn is not None and ccnn_n:
    geo = geo.join(comunidades(geo, ccnn, ccnn_n, 5))

# ── 4. Cruce con la producción (último CSV del notebook de producción) ──────
prod_csv = sorted([k for k in datos_drive if k.startswith("PP_produccion_lote_mes_") and k.endswith(".csv")])
prod = None
if prod_csv:
    p = os.path.join(ruta, prod_csv[-1]); bajar_de_drive(datos_drive[prod_csv[-1]], p)
    prod = pd.read_csv(p, encoding="utf-8-sig")
    acum = (prod[prod["PRODUCTO"] == "PETROLEO"].groupby("LOTE")["BARRILES"].sum() / 1e6).round(3)
    geo["PETROLEO_ACUM_MMBBL"] = geo["LOTE"].map(acum).fillna(0)
    # mismo polígono, varios contratos en el tiempo (p. ej. 1-AB + 192): acumulado del campo
    geo["PETROLEO_ACUM_CAMPO_MMBBL"] = geo.groupby("GEOMETRIA_DE")["PETROLEO_ACUM_MMBBL"].transform("sum").round(3)
    falta = sorted(set(prod["LOTE"].astype(str)) - set(geo["LOTE"]))
    print(f"\n🔗 Producción leída de {prod_csv[-1]}")
    print(f"   Lotes con producción y sin polígono ({len(falta)}): {', '.join(falta) if falta else 'ninguno'}")
    if falta:
        print("   → son lotes históricos o con otro nombre en el mapa; se completan a mano en LOTES_ALIAS_GEOMETRIA.")
    log.append({"CAPA": "LOTES SIN POLIGONO", "URL": "", "FUENTE": prod_csv[-1], "ENTIDADES": len(falta),
                "ESTADO": ", ".join(falta)[:500]})
else:
    print("\nℹ️ No encontré PP_produccion_lote_mes_*.csv en DATA: corre antes el notebook de producción para el cruce.")

# ── 5. Guardar ───────────────────────────────────────────────────────────────
cols = [c for c in ["LOTE", "GEOMETRIA_DE", "LOTE_MAPA", "OPERADOR_MAPA", "ESTADO_MAPA", "CAPA", "DEPARTAMENTO",
                    "DEPARTAMENTO_PCT", "DEPARTAMENTO_DETALLE", "DEPARTAMENTO_N", "ZOCALO", "CUENCA", "CUENCA_PCT",
                    "AREA_KM2", "LAT", "LON", "DIST_OLEODUCTO_KM", "DIST_IQUITOS_KM", "DIST_RIO_GRANDE_KM", "RIO_CERCANO",
                    "ANP_PCT", "ANP_DETALLE", "CCNN_DENTRO_N", "CCNN_5KM_N", "CCNN_DENTRO", "PETROLEO_ACUM_MMBBL", "PETROLEO_ACUM_CAMPO_MMBBL"]
        if c in geo.columns]
tabla = pd.DataFrame(geo[cols]).sort_values("LOTE")
tabla.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
p = os.path.join(ruta, "PP_LOTES_GEO.csv"); tabla.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_ID, datos_drive)
guardar_geojson(geo[cols + ["geometry"]], "PP_lotes_indicadores.geojson")
pc = os.path.join(ruta, "CONTROL PP mapa descarga.csv")
pd.DataFrame(log).assign(FECHA=date.today().strftime("%d/%m/%Y")).to_csv(pc, index=False, encoding="utf-8-sig")
subir(pc, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️  DATA/PP_LOTES_GEO.csv ({len(tabla)} lotes) · DATA/MAPA/PP_lotes_indicadores.geojson · CONTROL actualizado")

if "DEPARTAMENTO" in tabla:
    print("\n📍 Lotes por departamento:")
    display(tabla.groupby("DEPARTAMENTO").agg(LOTES=("LOTE", "count"), AREA_KM2=("AREA_KM2", "sum")).sort_values("LOTES", ascending=False))
    lor = tabla[tabla["DEPARTAMENTO"] == "LORETO"]
    if len(lor):
        print("\n🛢️ Loreto:")
        display(lor.sort_values("PETROLEO_ACUM_MMBBL" if "PETROLEO_ACUM_MMBBL" in lor else "LOTE", ascending=False).head(25))

In [ ]:
# ═══ BLOQUE 4 · MAPA INTERACTIVO ═══
import folium, math
m = folium.Map(location=[-5.5, -75.5], zoom_start=5, control_scale=True)
def simpl(g, tol=0.002):
    g = g.copy(); g["geometry"] = g.geometry.simplify(tol, preserve_topology=True); return g

if deps is not None:
    folium.GeoJson(simpl(deps, 0.005), name="Departamentos",
                   style_function=lambda f: {"color": "#555", "weight": 1, "fillOpacity": 0},
                   tooltip=folium.GeoJsonTooltip(["DEPARTAMENTO"])).add_to(m)
if anp is not None:
    folium.GeoJson(simpl(anp[[c for c in [anp_n] if c] + ["geometry"]]), name="Áreas Naturales Protegidas", show=False,
                   style_function=lambda f: {"color": "#2e7d32", "weight": 0.5, "fillColor": "#66bb6a", "fillOpacity": 0.25}).add_to(m)
if rios_q is not None:
    folium.GeoJson(simpl(rios_q[["DIS_AV_CMS", "geometry"]]), name="Ríos grandes",
                   style_function=lambda f: {"color": "#4a90d9", "weight": 1.5}).add_to(m)
if onp is not None:
    folium.GeoJson(onp[["geometry"]], name="Oleoducto",
                   style_function=lambda f: {"color": "#111", "weight": 3, "dashArray": "6 4"}).add_to(m)
if ccnn is not None and ccnn_n:
    fg = folium.FeatureGroup(name="Comunidades nativas", show=False)
    for _, r in ccnn.to_crs(4326).iterrows():
        if r.geometry is None or r.geometry.is_empty:
            continue
        pt = r.geometry.representative_point()
        folium.CircleMarker([pt.y, pt.x], radius=2, color="#8e24aa", fill=True, tooltip=str(r[ccnn_n])).add_to(fg)
    fg.add_to(m)

capa_l = geo[cols + ["geometry"]].copy()
vmax = float(capa_l.get("PETROLEO_ACUM_CAMPO_MMBBL", pd.Series([0])).max() or 0)
def color(v):
    if not v or vmax <= 0:
        return "#bdbdbd"
    t = math.log10(1 + v) / math.log10(1 + vmax)
    return ["#fee8c8", "#fdbb84", "#fc8d59", "#e34a33", "#b30000"][min(4, int(t * 5))]
capa_l = capa_l[capa_l["GEOMETRIA_DE"] == capa_l["LOTE"]] if "GEOMETRIA_DE" in capa_l else capa_l
tip = [c for c in ["LOTE", "OPERADOR_MAPA", "ESTADO_MAPA", "DEPARTAMENTO", "CUENCA", "AREA_KM2", "PETROLEO_ACUM_CAMPO_MMBBL",
                   "DIST_OLEODUCTO_KM", "DIST_IQUITOS_KM", "ANP_PCT", "CCNN_DENTRO_N"] if c in capa_l]
folium.GeoJson(simpl(capa_l, 0.001), name="Lotes (color = petróleo acumulado del campo, MM bbl)",
               style_function=lambda f: {"color": "#7f2704", "weight": 1, "fillOpacity": 0.6,
                                         "fillColor": color(f["properties"].get("PETROLEO_ACUM_CAMPO_MMBBL"))},
               tooltip=folium.GeoJsonTooltip(tip)).add_to(m)
folium.Marker([IQUITOS.y, IQUITOS.x], tooltip="Iquitos").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
p = os.path.join(ruta, "PP_mapa_lotes.html"); m.save(p)
subir(p, MAPA_FOLDER_ID, mapa_drive, mime="text/html")
print("☁️  Mapa guardado en DATA/MAPA/PP_mapa_lotes.html (ábrelo desde Drive con doble clic → Vista previa o descárgalo)")
m